# Teil 3: Modell

In diesem Notebook erstelle ich ein einfaches Modell, das vorhersagen soll, ob ein Kunde die Bank verlässt.


In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score


## Daten einlesen

Zuerst lese ich den Datensatz wieder mit pandas ein.


In [2]:
df = pd.read_csv('churn_modelling.csv')
df.head()


,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


## Daten vorbereiten

`RowNumber`, `CustomerId` und `Surname` helfen dem Modell nicht wirklich. Deshalb entferne ich diese Spalten. `Geography` und `Gender` wandle ich mit `get_dummies` in Zahlen um.


In [3]:
df_model = df.drop(columns=['RowNumber', 'CustomerId', 'Surname'])
df_model = pd.get_dummies(df_model, columns=['Geography', 'Gender'], drop_first=True)

y = df_model['Exited']
X = df_model.drop(columns=['Exited'])

print('Features:')
print(X.columns.tolist())
X.head()


Features:
['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Geography_Germany', 'Geography_Spain', 'Gender_Male']


,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_Germany,Geography_Spain,Gender_Male
0,619,42,2,0.00,1,1,1,101348.88,False,False,False
1,608,41,1,83807.86,1,0,1,112542.58,False,True,False
2,502,42,8,159660.80,3,1,0,113931.57,False,False,False
3,699,39,1,0.00,2,0,0,93826.63,False,False,False
4,850,43,2,125510.82,1,1,1,79084.10,False,True,False


## Train-Test-Split

Ich verwende 80 Prozent der Daten zum Trainieren und 20 Prozent zum Testen.


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print('Training:', X_train.shape)
print('Test:', X_test.shape)


Training: (8000, 11)
Test: (2000, 11)


## Gewählter Algorithmus

Ich verwende hier einen `RandomForestClassifier`. Der Algorithmus passt, weil das Ziel `Exited` nur zwei Werte hat: 0 oder 1. Ein Random Forest kann numerische und umgewandelte kategorische Felder gut zusammen benutzen. Er ist auch relativ einfach anzuwenden, weil man nicht zuerst sehr viele Einstellungen suchen muss. Für diese LB reicht mir ein normales Modell ohne Hyperparameter-Tuning.


In [5]:
model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)


RandomForestClassifier(random_state=42)

## Vorhersagen

Jetzt mache ich Vorhersagen mit dem Testdatensatz und vergleiche sie mit den echten Werten.


In [6]:
y_pred = model.predict(X_test)

vergleich = pd.DataFrame({
    'Echt': y_test.values,
    'Vorhersage': y_pred
})

vergleich.head(15)


,Echt,Vorhersage
0,0,0
1,0,0
2,0,0
3,0,0
4,0,0
5,0,0
6,0,0
7,1,0
8,0,0
9,0,0


In [7]:
accuracy = accuracy_score(y_test, y_pred)
print('Accuracy:', round(accuracy, 4))


Accuracy: 0.8665


## Manuelle Überprüfung

Beim Vergleich sieht man, dass das Modell viele Werte richtig trifft. Es sagt aber auch oft `0` voraus, weil die meisten Kunden im Datensatz nicht gekündigt haben. Das ist logisch, aber dadurch kann das Modell Kündigungen auch verpassen.


## Erkenntnisse

Das Modell erreicht eine brauchbare Genauigkeit, aber die Accuracy alleine sagt nicht alles. Der Datensatz ist nicht ganz ausgeglichen, weil deutlich mehr Kunden geblieben sind als gekündigt haben. Darum wirkt ein Modell schnell gut, wenn es oft `0` vorhersagt. Für eine echte Bank müsste man besonders prüfen, ob kündigende Kunden gut erkannt werden.
